In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import SegformerFeatureExtractor, SegformerForSemanticSegmentation
from PIL import Image
import numpy as np
import requests
from torchvision import transforms
from tqdm import tqdm

# First, we'll define the ConvBlock that's used in your StudentUNet
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        return x

# Student UNet model as provided
class StudentUNet(nn.Module):
    """
    Student model: Simplified UNet architecture with fewer parameters
    ~485,000 parameters (~64x smaller than teacher)
    """
    def __init__(self, in_channels=3, num_classes=19):  # Update to 19 classes for Cityscapes
        super(StudentUNet, self).__init__()
        
        # Encoder - fewer filters and simpler architecture
        self.enc1 = ConvBlock(in_channels, 32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.enc2 = ConvBlock(32, 64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.enc3 = ConvBlock(64, 128)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Bottleneck - smaller than teacher
        self.bottleneck = ConvBlock(128, 256)
        
        # Decoder - fewer filters and simpler architecture
        self.upconv3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(256, 128)  # 256 = 128 + 128 (skip connection)
        
        self.upconv2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(128, 64)  # 128 = 64 + 64 (skip connection)
        
        self.upconv1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(64, 32)  # 64 = 32 + 32 (skip connection)
        
        # Output layer
        self.output = nn.Conv2d(32, num_classes, kernel_size=1)
    
    def forward(self, x):
        # Encoder
        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool1(enc1))
        enc3 = self.enc3(self.pool2(enc2))
        
        # Bottleneck
        bottleneck = self.bottleneck(self.pool3(enc3))
        
        # Decoder with skip connections
        dec3 = self.upconv3(bottleneck)
        dec3 = torch.cat([dec3, enc3], dim=1)
        dec3 = self.dec3(dec3)
        
        dec2 = self.upconv2(dec3)
        dec2 = torch.cat([dec2, enc2], dim=1)
        dec2 = self.dec2(dec2)
        
        dec1 = self.upconv1(dec2)
        dec1 = torch.cat([dec1, enc1], dim=1)
        dec1 = self.dec1(dec1)
        
        # Output layer
        out = self.output(dec1)
        
        return out

# Simple dataset class for loading images (replace with your actual dataset)
class CityscapesDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        # For this example, we'll just load a sample image (replace with actual path loading)
        image = Image.open(requests.get(image_path, stream=True).raw).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        return image

# Knowledge Distillation Loss 
class DistillationLoss(nn.Module):
    def __init__(self, alpha=0.5, temperature=2.0):
        super(DistillationLoss, self).__init__()
        self.alpha = alpha  # Weight for distillation loss vs. standard loss
        self.temperature = temperature  # Temperature for softening probabilities
        self.kl_div = nn.KLDivLoss(reduction='batchmean')
        self.ce_loss = nn.CrossEntropyLoss()
        
    def forward(self, student_logits, teacher_probs, targets=None):
        """
        Args:
            student_logits: Output logits from student (before softmax)
            teacher_probs: Soft target probabilities from teacher (after softmax)
            targets: Optional hard labels for additional supervision
        """
        # Apply temperature scaling and softmax to student logits
        student_probs = torch.nn.functional.softmax(student_logits / self.temperature, dim=1)
        
        # Log softmax for KL divergence
        student_log_probs = torch.nn.functional.log_softmax(student_logits / self.temperature, dim=1)
        
        # Calculate KL divergence loss (distillation loss)
        # KL div wants log probabilities for the first input
        distillation_loss = self.kl_div(student_log_probs, teacher_probs) * (self.temperature ** 2)
        
        # Calculate standard CE loss if targets provided
        if targets is not None:
            ce_loss = self.ce_loss(student_logits, targets)
            # Weighted sum of the two losses
            return self.alpha * ce_loss + (1 - self.alpha) * distillation_loss
        
        return distillation_loss

def train_step(student_model, teacher_model, feature_extractor, images, optimizer, criterion, device):
    # Set models to proper mode
    student_model.train()
    teacher_model.eval()  # Teacher is always in eval mode
    
    # Move images to device
    images = images.to(device)
    batch_size, channels, height, width = images.shape
    
    # Get teacher predictions - no_grad to save memory and computations
    with torch.no_grad():
        # Process with feature extractor for teacher
        inputs = feature_extractor(
            images=images.permute(0, 2, 3, 1).cpu().numpy(), 
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Get teacher logits
        teacher_outputs = teacher_model(**inputs)
        teacher_logits = teacher_outputs.logits  # (B, num_classes, H/4, W/4)
        
        # Upsample teacher logits to match original image size
        teacher_logits_upsampled = torch.nn.functional.interpolate(
            teacher_logits,
            size=(height, width),
            mode="bilinear",
            align_corners=False
        )
        
        # Get soft probabilities from teacher
        teacher_probs = torch.nn.functional.softmax(teacher_logits_upsampled, dim=1)
    
    # Forward pass through student model
    student_logits = student_model(images)  # (B, num_classes, H, W)
    
    # Calculate distillation loss
    loss = criterion(student_logits, teacher_probs)
    
    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    return loss.item()

def train_distillation(student_model, teacher_model, feature_extractor, dataloader, num_epochs=10, device="cuda"):
    """Main training loop for knowledge distillation"""
    student_model = student_model.to(device)
    teacher_model = teacher_model.to(device)
    
    # Initialize optimizer and loss
    optimizer = optim.Adam(student_model.parameters(), lr=1e-4)
    criterion = DistillationLoss(alpha=0.1, temperature=3.0)  # Emphasize teacher's knowledge
    
    for epoch in range(num_epochs):
        total_loss = 0.0
        
        # Training loop
        progress_bar = tqdm(dataloader)
        for images in progress_bar:
            loss = train_step(student_model, teacher_model, feature_extractor, 
                             images, optimizer, criterion, device)
            total_loss += loss
            
            # Update progress bar
            progress_bar.set_description(f"Epoch {epoch+1}/{num_epochs}")
            progress_bar.set_postfix(loss=loss)
        
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")
    
    return student_model

def main():
    # Check for GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Load teacher model (SegFormer)
    feature_extractor = SegformerFeatureExtractor.from_pretrained("nvidia/segformer-b5-finetuned-cityscapes-1024-1024")
    teacher_model = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b5-finetuned-cityscapes-1024-1024")
    
    # Create student model
    # Cityscapes has 19 classes
    student_model = StudentUNet(in_channels=3, num_classes=19)
    
    # Print model sizes to confirm the student is smaller
    teacher_params = sum(p.numel() for p in teacher_model.parameters())
    student_params = sum(p.numel() for p in student_model.parameters())
    print(f"Teacher parameters: {teacher_params:,}")
    print(f"Student parameters: {student_params:,}")
    print(f"Compression ratio: {teacher_params / student_params:.1f}x")
    
    # Sample image URLs for demonstration (replace with your dataset)
    image_urls = [
        "http://images.cocodataset.org/val2017/000000039769.jpg",
        # Add more image URLs here
    ]
    
    # Create dataset and dataloader
    transform = transforms.Compose([
        transforms.Resize((512, 512)),  # Resize for consistent processing
        transforms.ToTensor(),
    ])
    
    dataset = CityscapesDataset(image_urls, transform=transform)
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True)
    
    # Train using distillation
    trained_student = train_distillation(
        student_model=student_model,
        teacher_model=teacher_model,
        feature_extractor=feature_extractor,
        dataloader=dataloader,
        num_epochs=5,
        device=device
    )
    
    # Save the trained student model
    torch.save(trained_student.state_dict(), "student_cityscapes_model.pth")
    print("Training complete. Model saved.")

if __name__ == "__main__":
    main()